##
Global Token Efficiency Data Collection Pipeline
================================================
Collects place names from OpenStreetMap via Overpass API for all H3 cells
at resolution 3 worldwide, and computes token efficiency ratios for
GPT-4o, DeepSeek-LLM-7B, and Mistral-7B tokenizers.

Output: ResultsWithAllThreeRatios.json


In [ ]:
# ── 1. IMPORTS ────────────────────────────────────────────────────────────────
import json
import time
import os
import overpy
import h3
import tiktoken
from transformers import AutoTokenizer

# ── 2. CONFIGURATION ──────────────────────────────────────────────────────────
RESOLUTION = 3
PLACE_LIMIT = 100
NATURAL_LIMIT = 110
OTHER_LIMIT = 90
SLEEP_BETWEEN_CELLS = 1       # seconds between Overpass requests
SLEEP_ON_ERROR = 60           # seconds to wait after rate limit error
LOG_EVERY_N = 100             # print progress every N cells
MIN_ENTRIES = 0               # keep all cells, filter later in analysis
OUTPUT_PATH = "ResultsWithAllThreeRatios.json"
CHECKPOINT_PATH = "checkpoint.json"  # resume support
HF_TOKEN = "your_huggingface_token_here"  # replace with your token

# ── 3. BOUNDING BOX: ENTIRE WORLD ────────────────────────────────────────────
WEST, SOUTH, EAST, NORTH = -175, -60, 165, 80

# ── 4. TOKENIZER SETUP ───────────────────────────────────────────────────────
print("Loading tokenizers...")

# GPT-4o tokenizer via TikToken
enc_gpt = tiktoken.encoding_for_model("gpt-4o")

# Mistral-7B tokenizer via Hugging Face
tokenizer_mistral = AutoTokenizer.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.2",
    use_fast=True,
    token=HF_TOKEN
)

# DeepSeek-LLM-7B tokenizer via Hugging Face
tokenizer_deepseek = AutoTokenizer.from_pretrained(
    "deepseek-ai/deepseek-llm-7b-base",
    use_fast=True
)

print("Tokenizers loaded.")

# ── 5. OVERPASS API SETUP ─────────────────────────────────────────────────────
api = overpy.Overpass()

# ── 6. H3 CELL GENERATION ────────────────────────────────────────────────────
print("Generating global H3 cells...")

world_poly = [
    (SOUTH, WEST), (SOUTH, EAST),
    (NORTH, EAST), (NORTH, WEST),
    (SOUTH, WEST)
]
geojson = {
    "type": "Polygon",
    "coordinates": [[(lon, lat) for lat, lon in world_poly]]
}
all_cells = list(h3.polyfill(geojson, RESOLUTION, geo_json_conformant=True))
print(f"Total H3 cells at resolution {RESOLUTION}: {len(all_cells)}")

# ── 7. HELPER FUNCTIONS ───────────────────────────────────────────────────────

def compute_avg_token_ratio(names, tokenize_fn):
    """
    Compute average token ratio across a list of names.
    Token ratio = word count / token count.
    Returns 0 if no names are provided.
    """
    ratios = []
    for name in names:
        word_count = len(name.split())
        token_count = tokenize_fn(name)
        if token_count > 0:
            ratios.append(word_count / token_count)
    return round(sum(ratios) / len(ratios), 3) if ratios else 0


def tokenize_gpt(name):
    return len(enc_gpt.encode(name))


def tokenize_mistral(name):
    return len(tokenizer_mistral.encode(name))


def tokenize_deepseek(name):
    return len(tokenizer_deepseek.encode(name))


def query_cell(cell):
    """
    Query Overpass API for place names in an H3 cell.
    Returns categorized and limited node lists.
    """
    boundary = h3.h3_to_geo_boundary(cell, geo_json=True)
    poly_str = " ".join(f"{lat} {lon}" for lon, lat in boundary)

    query = f"""
    [out:json][timeout:60];
    (
      node["name"]["place"](poly:"{poly_str}");
      node["name"]["natural"](poly:"{poly_str}");
      node["name"]["waterway"](poly:"{poly_str}");
      node["name"](poly:"{poly_str}");
    );
    out center qt 300;
    """
    return api.query(query)


def process_cell(cell):
    """
    Query, categorize, limit, and tokenize names for a single H3 cell.
    Returns a dict with entries and avg token ratios for all three models.
    """
    # query with error handling
    try:
        result = query_cell(cell)
    except Exception as e:
        print(f"  Error querying cell {cell}: {e}. Retrying after {SLEEP_ON_ERROR}s...")
        time.sleep(SLEEP_ON_ERROR)
        result = query_cell(cell)

    # categorize nodes
    place_nodes = [n for n in result.nodes if n.tags.get("place")]
    natural_nodes = [n for n in result.nodes
                     if n.tags.get("natural") or n.tags.get("waterway")]
    other_nodes = [n for n in result.nodes
                   if "name" in n.tags
                   and not n.tags.get("place")
                   and not n.tags.get("natural")
                   and not n.tags.get("waterway")]

    # apply limits
    place_nodes = place_nodes[:PLACE_LIMIT]
    natural_nodes = natural_nodes[:NATURAL_LIMIT]
    other_nodes = other_nodes[:OTHER_LIMIT]
    selected = place_nodes + natural_nodes + other_nodes

    # build entries
    entries = []
    for node in selected:
        name = node.tags.get("name", "")
        if not name:
            continue
        entries.append({
            "name": name,
            "category": (
                "place" if node in place_nodes
                else "natural" if node in natural_nodes
                else "other"
            ),
            "lat": float(node.lat),
            "lon": float(node.lon),
        })

    # extract name list
    names = [e["name"] for e in entries]

    # compute avg token ratios
    chatgpt_avg = compute_avg_token_ratio(names, tokenize_gpt)
    mistral_avg = compute_avg_token_ratio(names, tokenize_mistral)
    deepseek_avg = compute_avg_token_ratio(names, tokenize_deepseek)

    return {
        "entries": entries,
        "chatgpt_avg_token_ratio": chatgpt_avg,
        "mistral_avg_token_ratio": mistral_avg,
        "deepseek_avg_token_ratio": deepseek_avg
    }

# ── 8. RESUME SUPPORT ─────────────────────────────────────────────────────────

def load_checkpoint():
    """Load existing results if a checkpoint file exists."""
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH, "r", encoding="utf-8") as f:
            data = json.load(f)
        print(f"Resuming from checkpoint: {len(data)} cells already processed.")
        return data
    return {}


def save_checkpoint(data):
    """Save current results to checkpoint file."""
    with open(CHECKPOINT_PATH, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False)

# ── 9. MAIN PROCESSING LOOP ───────────────────────────────────────────────────

results = load_checkpoint()
already_done = set(results.keys())
remaining_cells = [c for c in all_cells if c not in already_done]

print(f"Cells to process: {len(remaining_cells)} "
      f"(skipping {len(already_done)} already done)")

for idx, cell in enumerate(remaining_cells, start=1):

    cell_data = process_cell(cell)
    results[cell] = cell_data

    # save checkpoint every 100 cells
    if idx % LOG_EVERY_N == 0:
        save_checkpoint(results)
        print(f"Progress: {len(already_done) + idx} / {len(all_cells)} cells "
              f"({(len(already_done) + idx) / len(all_cells) * 100:.1f}%)")

    time.sleep(SLEEP_BETWEEN_CELLS)

# ── 10. FINAL EXPORT ──────────────────────────────────────────────────────────

# filter out empty cells
filtered = {
    cell: info for cell, info in results.items()
    if isinstance(info.get("entries"), list) and len(info["entries"]) > 0
}

removed = len(results) - len(filtered)
print(f"\nRemoved {removed} empty cells.")
print(f"Final dataset: {len(filtered)} cells.")

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(filtered, f, ensure_ascii=False, indent=2)

print(f"Saved to {OUTPUT_PATH}")

# clean up checkpoint after successful completion
if os.path.exists(CHECKPOINT_PATH):
    os.remove(CHECKPOINT_PATH)
    print("Checkpoint removed.")

## Name Count Visualization

Loads OSM.json, 

counts place names per H3 cell, 

computes global and regional statistics, 

and produces a classified world map showing name density per cell.

Input:  OSM.json
Output: world_names_per_h3_classed.png
        world_names_per_h3_classed.pdf
        global_stats.csv
        region_stats.csv

In [ ]:
# ── 1. IMPORTS ────────────────────────────────────────────────────────────────
import json
import h3
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap, BoundaryNorm
from shapely.geometry import Polygon

# ── 2. CONFIGURATION ──────────────────────────────────────────────────────────
OSM_PATH = "OSM.json"
OUT_PNG   = "world_names_per_h3_classed.png"
OUT_PDF   = "world_names_per_h3_classed.pdf"

# ── 3. LOAD DATA ──────────────────────────────────────────────────────────────
with open(OSM_PATH, "r", encoding="utf-8") as f:
    osm = json.load(f)

# ── 4. COUNT NAMES PER CELL ───────────────────────────────────────────────────
def count_names(entries):
    """Count number of place name entries for a cell."""
    if isinstance(entries, dict) and "entries" in entries:
        return len(entries.get("entries", []))
    elif isinstance(entries, list):
        return len(entries)
    return 0

cell_counts = {
    cell: count_names(vals)
    for cell, vals in osm.items()
    if count_names(vals) > 0
}

df = pd.DataFrame([
    {"cell": cell, "count": cnt}
    for cell, cnt in cell_counts.items()
])

# ── 5. REGION ASSIGNMENT ──────────────────────────────────────────────────────
def get_region(cell):
    """Assign a broad geographic region based on cell centroid coordinates."""
    lat, lon = h3.h3_to_geo(cell)
    if -170 <= lon <= -30 and -60 <= lat <= 80:
        return "Americas"
    elif -30 < lon <= 60 and 0 <= lat <= 70:
        return "Europe"
    elif 60 < lon <= 180 and 0 <= lat <= 80:
        return "Asia"
    elif -30 < lon <= 60 and -40 <= lat < 0:
        return "Africa"
    elif 110 <= lon <= 180 and -50 <= lat < 0:
        return "Oceania"
    else:
        return "Other"

df["region"] = df["cell"].apply(get_region)

# ── 6. STATISTICS ─────────────────────────────────────────────────────────────
global_stats = df["count"].describe()
region_stats = df.groupby("region")["count"].describe()

print("Global statistics:")
print(global_stats)
print("\nPer-region statistics:")
print(region_stats)

global_stats.to_csv("global_stats.csv")
region_stats.to_csv("region_stats.csv")
print("\nStatistics saved to global_stats.csv and region_stats.csv")

# ── 7. BUILD GEODATAFRAME ─────────────────────────────────────────────────────
polys = []
for cell in df["cell"]:
    boundary = h3.h3_to_geo_boundary(cell, geo_json=True)  # returns (lon, lat)
    polys.append(Polygon(boundary))

gdf = gpd.GeoDataFrame(
    {"cell": df["cell"], "count": df["count"].values},
    geometry=polys,
    crs="EPSG:4326"
)

# ── 8. CLASSIFY INTO THREE GROUPS ─────────────────────────────────────────────
def classify(v):
    if v < 100:
        return 0
    elif v < 200:
        return 1
    else:
        return 2

gdf["class_id"] = gdf["count"].apply(classify)

# ── 9. WORLD BOUNDARIES ───────────────────────────────────────────────────────
world = gpd.read_file(
    "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
).to_crs("EPSG:4326")
world = world[world["CONTINENT"] != "Antarctica"]

# ── 10. PLOT ──────────────────────────────────────────────────────────────────
cmap = ListedColormap(["#d73027", "#fee08b", "#1a9850"])
norm = BoundaryNorm([0, 1, 2, 3], cmap.N)

fig, ax = plt.subplots(figsize=(12, 6))

world.plot(ax=ax, color="white", edgecolor="lightgrey", linewidth=0.5)

gdf.plot(
    ax=ax,
    column="class_id",
    cmap=cmap,
    norm=norm,
    linewidth=0,
    alpha=0.95
)

ax.set_axis_off()
ax.set_xlim(-180, 165)
ax.set_ylim(-60, 80)

# legend
handles = [
    mpatches.Patch(facecolor=cmap.colors[0], edgecolor="none", label="< 100"),
    mpatches.Patch(facecolor=cmap.colors[1], edgecolor="none", label="100\u2013200"),
    mpatches.Patch(facecolor=cmap.colors[2], edgecolor="none", label="\u2265 200"),
]
leg = ax.legend(
    handles=handles,
    title="Names per H3 cell",
    loc="lower left",
    bbox_to_anchor=(0.02, 0.04),
    frameon=True,
    fancybox=False,
    framealpha=1.0,
    borderpad=0.6,
    labelspacing=0.35,
    handlelength=1.0,
    handleheight=0.9,
    fontsize=9,
    title_fontsize=9
)
frame = leg.get_frame()
frame.set_edgecolor("black")
frame.set_linewidth(0.8)
frame.set_facecolor("white")

ax.text(
    0.99, 0.02, "EPSG:4326",
    transform=ax.transAxes, ha="right", va="bottom",
    fontsize=8, color="grey"
)

# ── 11. EXPORT ────────────────────────────────────────────────────────────────
fig.savefig(OUT_PNG, dpi=400, bbox_inches="tight")
fig.savefig(OUT_PDF, bbox_inches="tight")
plt.show()
print(f"Saved: {OUT_PNG} and {OUT_PDF}")

## Create Maps

Loads ResultsWithAllThreeRatios.json and produces four world maps:
- GPT-4o token ratio
- DeepSeek-LLM-7B token ratio
- Mistral-7B token ratio
- Mean token ratio across all three models

Input:  ResultsWithAllThreeRatios.json
Output: ratio_chatgpt_epsg4326.png / .pdf
        ratio_deepseek_epsg4326.png / .pdf
        ratio_mistral_epsg4326.png / .pdf
        ratio_avg_3models_epsg4326.png / .pdf

In [ ]:
# ── 1. IMPORTS ────────────────────────────────────────────────────────────────
import json
import math
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon
import h3
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable

# ── 2. CONFIGURATION ──────────────────────────────────────────────────────────
JSON_PATH   = "ResultsWithAllThreeRatios.json"
MIN_ENTRIES = 100
WEST, SOUTH, EAST, NORTH = -175, -60, 165, 80

MAPS = {
    "chatgpt": {
        "key":     "chatgpt_avg_token_ratio",
        "out_png": "ratio_chatgpt_epsg4326.png",
        "out_pdf": "ratio_chatgpt_epsg4326.pdf",
        "label":   "GPT-4o",
    },
    "deepseek": {
        "key":     "deepseek_avg_token_ratio",
        "out_png": "ratio_deepseek_epsg4326.png",
        "out_pdf": "ratio_deepseek_epsg4326.pdf",
        "label":   "DeepSeek-LLM-7B",
    },
    "mistral": {
        "key":     "mistral_avg_token_ratio",
        "out_png": "ratio_mistral_epsg4326.png",
        "out_pdf": "ratio_mistral_epsg4326.pdf",
        "label":   "Mistral-7B",
    },
    "avg": {
        "key":     None,  # computed from all three
        "out_png": "ratio_avg_3models_epsg4326.png",
        "out_pdf": "ratio_avg_3models_epsg4326.pdf",
        "label":   "Mean of three models",
    },
}

# ── 3. LOAD DATA ──────────────────────────────────────────────────────────────
with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Total cells in JSON: {len(data)}")

# ── 4. WORLD BOUNDARIES (loaded once) ────────────────────────────────────────
world = gpd.read_file(
    "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
).to_crs("EPSG:4326")
world = world[world["CONTINENT"] != "Antarctica"]

# ── 5. COLORMAP AND NORM (shared across all maps) ────────────────────────────
cmap = LinearSegmentedColormap.from_list(
    "ylgnbu_like",
    ["#f7fcf0", "#ccebc5", "#7bccc4", "#2b8cbe", "#084081"]
)
norm = Normalize(vmin=0, vmax=1)

# ── 6. HELPER: BUILD GEODATAFRAME FOR A GIVEN RATIO KEY ──────────────────────
def build_gdf(data, ratio_key):
    """
    Build a GeoDataFrame of H3 cell polygons with token ratio values.
    If ratio_key is None, compute the mean across all three models.
    """
    rows = []
    for cell, info in data.items():
        entries = info.get("entries")
        if not isinstance(entries, list) or len(entries) < MIN_ENTRIES:
            continue

        if ratio_key is None:
            vals = [
                info.get("chatgpt_avg_token_ratio"),
                info.get("mistral_avg_token_ratio"),
                info.get("deepseek_avg_token_ratio"),
            ]
            vals = [v for v in vals
                    if isinstance(v, (int, float)) and not math.isnan(v)]
            if not vals:
                continue
            ratio = sum(vals) / len(vals)
        else:
            ratio = info.get(ratio_key)
            if not isinstance(ratio, (int, float)) or math.isnan(ratio):
                continue

        rows.append((cell, float(ratio)))

    if not rows:
        raise ValueError(
            f"No cells remaining for ratio_key={ratio_key}. "
            "Check MIN_ENTRIES or JSON structure."
        )

    df = pd.DataFrame(rows, columns=["cell", "ratio"])
    polys = [
        Polygon(h3.h3_to_geo_boundary(cell, geo_json=True))
        for cell in df["cell"]
    ]
    gdf = gpd.GeoDataFrame(df, geometry=polys, crs="EPSG:4326")
    gdf["ratio"] = gdf["ratio"].clip(0, 1)
    return gdf

# ── 7. HELPER: PLOT AND SAVE MAP ─────────────────────────────────────────────
def plot_map(gdf, out_png, out_pdf):
    """
    Plot a world map of token ratios and save to PNG and PDF.
    """
    fig, ax = plt.subplots(figsize=(12, 6))

    ax.set_facecolor("#d7dedf")

    world.plot(
        ax=ax,
        color="white",
        edgecolor="#cfcfcf",
        linewidth=0.25,
        zorder=1
    )

    gdf.plot(
        ax=ax,
        column="ratio",
        cmap=cmap,
        norm=norm,
        alpha=0.60,
        linewidth=0.0,
        edgecolor="none",
        antialiased=True,
        rasterized=True,
        zorder=2
    )

    ax.set_axis_off()
    ax.set_xlim(WEST, EAST)
    ax.set_ylim(SOUTH, NORTH)

    # colorbar
    sm = ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([])
    cax = fig.add_axes([0.12, 0.11, 0.22, 0.02])
    cbar = fig.colorbar(sm, cax=cax, orientation="horizontal")
    cbar.set_ticks([0, 0.25, 0.5, 0.75, 1.0])
    cbar.ax.tick_params(labelsize=9)

    for spine in cax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.8)
        spine.set_edgecolor("black")
    cax.set_facecolor("white")

    ax.text(
        0.99, 0.02, "EPSG:4326",
        transform=ax.transAxes, ha="right", va="bottom",
        fontsize=8, color="grey"
    )

    fig.subplots_adjust(left=0.01, right=0.99, top=0.98, bottom=0.05)

    fig.savefig(out_png, dpi=500, bbox_inches="tight")
    fig.savefig(out_pdf, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_png} and {out_pdf}")

# ── 8. GENERATE ALL FOUR MAPS ─────────────────────────────────────────────────
for name, config in MAPS.items():
    print(f"\nBuilding map: {config['label']}...")
    gdf = build_gdf(data, config["key"])
    print(f"  Cells included: {len(gdf)}")
    plot_map(gdf, config["out_png"], config["out_pdf"])

print("\nAll maps saved.")